# Monte Carlo reinforcement learning

_Artificial Intelligence — more_

**No model? Just play out whole episodes and average the rewards you actually got.**

Q-learning bootstraps off its own estimates. Monte Carlo RL (Reinforcement Learning) does something even simpler: it just averages real outcomes.

     Play a full episode to the end. Add up the discounted rewards you actually received: that is the return $u$.

---

This notebook is a practice scaffold for the **Monte Carlo reinforcement learning** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## Reference implementation — Python + NumPy

### Step 1 — Simulate one episode's return

Monte Carlo estimates a value by *playing out whole episodes* and averaging what actually happened. Here each episode reaches the goal in a random 2–4 steps, paying `-1` per step and `+10` at the goal.

The **return** $u$ is the sum of those rewards, each discounted by a growing factor `disc` (which is `gamma` raised to the step number) so rewards further in the future count for less.

In [ ]:
rng = np.random.default_rng(0)

gamma = 0.9   # discount: later rewards are worth less than immediate ones

def sample_return():
    # Reach the goal in 2..4 random steps: -1 per step, then +10 at the goal.
    steps = rng.integers(2, 5)

    u = 0.0      # accumulated discounted return for this episode
    disc = 1.0   # current discount factor, starts at gamma^0 = 1

    for _ in range(steps):
        step_reward = -1            # cost of taking one step
        u += disc * step_reward     # add the discounted step reward
        disc *= gamma               # shrink the discount for the next step

    goal_reward = 10                # reward collected at the goal
    u += disc * goal_reward         # add the discounted goal reward
    return u

### Step 2 — Average many returns into a running estimate

With no model of the environment, our value estimate $\hat{Q}$ is simply the **average of the returns** we have seen so far. We sample 2000 independent episodes and take a *running* mean.

`np.cumsum` gives the cumulative sum of returns; dividing by the episode count `1, 2, 3, ...` turns each prefix sum into the average-so-far, letting us watch the estimate settle as more episodes accumulate.

In [ ]:
returns = [sample_return() for _ in range(2000)]

# Running average: cumulative sum of returns divided by the count so far.
counts = np.arange(1, len(returns) + 1)
running = np.cumsum(returns) / counts

print("first 3 returns:", [round(r, 2) for r in returns[:3]])
print("Q-hat after   10 episodes:", round(running[9], 3))
print("Q-hat after  100 episodes:", round(running[99], 3))
print("Q-hat after 2000 episodes:", round(running[-1], 3))

## Visualize the data & results

_Blackjack: by averaging real hand outcomes, what is the value of standing on 20 against a dealer showing 6?_

### Step 1 — Define the blackjack environment

Now a more realistic example: estimate the value of **standing on 20 against a dealer showing 6**, purely by averaging real hand outcomes.

`draw` deals a card with face cards capped at 10 and an ace as 1. `dealer_play` makes the dealer hit until reaching 17 or more, treating an ace as 11 when it fits (a "soft" hand) and dropping it back to 1 if the total would otherwise bust.

In [ ]:
rng = np.random.default_rng(7)

# Blackjack: estimate the value of STANDING on player sum 20 vs dealer showing 6.
def draw():
    c = rng.integers(1, 14)
    if c == 1:
        return 1                 # ace counts as 1 here (soft handling is below)
    return min(c, 10)            # face cards (11,12,13) all count as 10

def dealer_play(showing):
    total = showing
    ace = (showing == 1)
    if ace:
        total += 10              # treat the ace as 11 while it fits
    while total in range(17):    # hit on totals 0..16, stand on 17+
        c = draw()
        if c == 1 and total + 11 in range(22):
            total += 11          # this ace fits as 11
            ace = True
        else:
            total += c
        while total > 21 and ace:
            total -= 10          # bust with a soft ace: demote it from 11 to 1
            ace = False
    return total

### Step 2 — Score one hand from the player's view

The player stands on 20, so the only thing left is to play out the dealer and compare. One episode returns `+1` if the player wins, `-1` if the dealer wins, and `0` for a push.

The dealer wins only by reaching exactly 21 (the single total above 20 that isn't a bust); a dealer total of 20 ties, and anything below 20 or above 21 hands the player the win.

In [ ]:
def episode():
    dealer = dealer_play(6)             # dealer reveals a 6, then hits to 17+
    if dealer > 21 or dealer in range(20):
        return 1.0                      # dealer busts or finishes below 20 -> player wins
    if dealer > 20:
        return -1.0                     # dealer made 21 -> player loses
    return 0.0                          # dealer made 20 -> push (tie)

### Step 3 — Simulate many hands and plot the running value

We play 5000 hands and, just like the toy example, take a running average of the outcomes. Plotted on a log x-axis, the estimate is noisy after a handful of hands but tightens toward its converged value (~0.70) as the sample grows — the Monte Carlo law of large numbers in action.

In [ ]:
N = 5000
rets = np.array([episode() for _ in range(N)])

# Running average return after each hand.
running = np.cumsum(rets) / np.arange(1, N + 1)

# Sample the running estimate at a few milestones for the plot.
xs = [1, 5, 10, 25, 50, 100, 250, 500, 1000, 2500, 5000]
ys = [running[x - 1] for x in xs]

fig, ax = plt.subplots()
ax.plot(xs, ys, "-o", color="#4ea1ff", label="running estimate")
ax.axhline(running[-1], color="#7ee787", label="converged value ~ 0.70")
ax.set_xscale("log")
ax.set_xlabel("blackjack hands simulated")
ax.set_ylabel("average return (+1 win, -1 loss)")
ax.set_title("Monte Carlo value of 'stand on 20 vs dealer 6' as hands accumulate")
ax.legend()
plt.show()